<a href="https://colab.research.google.com/github/addone/datascience-gt/blob/main/TM_wordcloud_practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 準備

内閣府[「景気ウォッチャー調査」](https://www5.cao.go.jp/keizai3/watcher/watcher_menu.html)データを利用し、街角の景気実感（タクシー運転手や小売店主の声）をテキスト化したものをテキストマイニング手法で分析しましょう。

### １、データのダウンロード

[調査概要](https://www5.cao.go.jp/keizai3/watcher/watcher_mokuteki.html#mokuteki)を確認してから、「公表資料（統計表一覧）」から最新の月次統計表（csv形式）をダウンロードします。

Excelなどでデータを確認し、「追加説明及び具体的状況の説明」列に注目させます。

### ２、データファイルをアップロード

ダウンロードしたcsvファイルをColabの「ファイル」にアップロードしてください。
（*接続を解除したら自動で削除されるため、使うたびにアップロードが必要です。*）

### ３、必要なツールのダウンロードとインストール

日本語文章を扱うために、形態素解析器[janome](https://janome.mocobeta.dev/ja/)、日本語フォントおよびグラフへ日本語出力用ツールをインストールします。

In [ ]:
!pip install janome
!apt-get -y install fonts-ipafont-gothic
!pip install japanize-matplotlib

---

## ステップ1：データ読み込み

Pythonでのデータ分析における業界標準ライブラリ[Pandas](https://pandas.pydata.org/)を用いて、データを読み込み、中身を確認しましょう。

参考資料：[データ分析で必須のPandasを入門しよう](https://aiacademy.jp/media/?p=152)

In [ ]:
import pandas as pd

# データの読み込み（内閣府からダウンロードしたCSVファイル）
df = pd.read_csv(______)

# データの中身を見てみる
print("--- 元のデータ ---")
df.head()

----

## ステップ2：テキスト抽出と形態素解析

まずはテキストマイニング用に文章を抽出します。

In [ ]:
# 注目される列のデータだけを取り出してリストにします
text_list = ______

# 欠損値（空欄）があるとうまくいかないので、文字列型に変換して結合
text_data = ' '.join(map(str, text_list))

# 最初の100文字だけを確認
print("▼ 分析対象のテキストデータ:")
print(______)

次に、形態素解析（文章を単語にバラバラにする）を行い、名詞と形容詞だけを取り出します。

In [ ]:
from janome.tokenizer import Tokenizer
tokenizer = Tokenizer()

words_list = []

# 名詞と形容詞だけを取り出す
for token in tokenizer.tokenize(text_data):
    if token.part_of_speech.split(',')[0] in [______]:
        # 「する」「いる」などの一般的すぎる言葉を除外（ストップワード）
        if token.base_form not in ["こと", "ため", "よう", "いる", "ある", "する"]:
            words_list.append(token.base_form)

# 最初の10単語だけを確認
print("抽出された単語:", ____, "...")

----

## ステップ3：ワードクラウドの生成

[WordCloud](https://amueller.github.io/word_cloud/)を使えば、テキストデータを可視化することができる。

参考資料：[ワードクラウドとは](https://data-viz-lab.com/word-cloud)

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import japanize_matplotlib

# ワードクラウドの生成設定
wc = WordCloud(
    background_color="white",
    width=800,
    height=600,
    font_path="fonts-japanese-mincho.ttf", # 日本語フォント指定
    colormap="winter" # 色味（cool, spring, autumnなど変更可）
)

# 単語リストを文字列に変換し、ワードクラウドを生成
text_for_cloud = ' '.join(____)
wc.generate(____)

# グラフ描画
plt.figure(figsize=(10, 8))
plt.imshow(wc)
plt.axis("off") # 軸を消す
plt.title(______)
plt.show()

----

## ステップ4：地域別のワードクラウドを比較

地域ごとのテキストデータを用意すれば、それぞれのワードクラウドを並べて作成できる。

まずは、用意したデータをそれぞれ読み込みます。

In [ ]:
# ２つの地域のテキストデータを読み込み、それぞれのワードクラウドを作成
df1 = pd.read_csv(______)
df2 = pd.read_csv(______)

# データの中身を見てみる
print("--- 元のデータ1 ---")
df1.head()

次に、地域ごとに単語リストを作成します。

In [ ]:
# 地域ごとにテキスト抽出と形態素解析する関数
def get_words(df):
    text_list = ______
    text_data = ' '.join(map(str, text_list))
    words = []

    # 名詞と形容詞だけを取り出す
    for token in tokenizer.tokenize(text_data):
        if token.part_of_speech.split(',')[0] in [______]:
            # 「する」「いる」などの一般的すぎる言葉を除外（ストップワード）
            if token.base_form not in ["こと", "ため", "よう", "いる", "ある", "する"]:
                words.append(token.base_form)

    return words

# それぞれの単語リストを作成
words_list1 = get_words(____)
words_list2 = get_words(____)

print("関東（南関東）のテキストから抽出された単語:", words_list1[:10], "...")
print("関西（近畿）のテキストから抽出された単語:", words_list2[:10], "...")

2つのワードクラウドを左右に並べて描画し、異なる地域のテキストデータを可視化して比較します。

In [ ]:
# 図の枠組みを作る（1行2列）
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
fpath = "fonts-japanese-mincho.ttf"

# --- 関東の描画 (左側: axes[0]) ---
text_kanto = ' '.join(____) # 単語リストを文字列に変換し
wc_kanto = WordCloud(background_color="white", width=600, height=600, font_path=fpath, colormap="winter") # 関東はクールな色
wc_kanto.generate(____)
axes[0].imshow(wc_kanto)
axes[0].set_title(______) # 左グラフのタイトル
axes[0].axis("off")

# --- 関西の描画 (右側: axes[1]) ---
text_kansai = ' '.join(____) # 単語リストを文字列に変換し
wc_kansai = WordCloud(background_color="white", width=600, height=600, font_path=fpath, colormap="autumn") # 関西は活気ある色
wc_kansai.generate(____)
axes[1].imshow(wc_kansai)
axes[1].set_title(______) # 右グラフのタイトル
axes[1].axis("off")

plt.suptitle(______) # グラフ全体タイトル
plt.show()

同じやり方で、例えば「飲食業 vs 製造業」や「2020年 vs 2024年」など、あらゆる比較が可能になります。